<a href="https://colab.research.google.com/github/AsiimweJr/LEARNING-Python_languages/blob/main/assignment1_individual_TB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Individual Assignment 1: Who is at risk of an unsuccessful TB treatment outcome?

**Fundamentals of Health Data Science**

### The situation

You work with the TB clinic of a district hospital. When a patient **starts** TB treatment, the clinic records some information about them. Six to eight months later, the treatment ends in one of several ways. Sadly, some patients **die**, are **lost to follow-up** (stop coming) or their treatment **fails**. Together these are called an **unsuccessful treatment outcome**.

The TB focal person can give extra support (home visits, phone follow-up, help with transport) to a limited number of patients. She asks:

> *"On the day a patient starts treatment, can we tell who is most at risk of an unsuccessful outcome, so we support them from the start?"*

The answer is **yes or no** (unsuccessful / successful), so this is a **classification** problem. You will build a **logistic regression** model, following the same journey as the demonstration notebook, but **you write all the code and make all the decisions.**

### Your data is personal

Everyone receives the same file, `tb_clinic_register.csv` (about 6,000 patients). At the start, you type your **registration number**. The notebook then picks **your own sample of 2,500 patients** and your own shuffle. Your numbers will differ from your classmates': **two students cannot have the same results.**

> The data are **simulated for teaching**. No real patients are in this file.

### Before you start: read this carefully

**This is an individual assignment.**
- Your code, your decisions and your written answers must be **your own work**.
- All written answers (*What you decided* and the ✍️ reflection questions) must be **in your own words. Do not use AI tools to write them.**
- **Any use of AI tools must be stated** in the declaration at the end: what you used it for, and where.
- If your submission suggests you may not understand your own work, you may be asked to complete a **short timed test or oral check** on your notebook, to confirm the work is yours.
- At the end you will declare that you have read through all your code, that it represents what you meant to do, and that you can **explain and defend every decision and every line**.

**About the reflection questions (✍️).**
They are **part of your mark**. They have **no single correct answer**: you are marked on how clearly you explain and defend **your own** decisions using **your own** results. They are **not** past or future exam questions, and you should not expect to see them in the exam.

**How this notebook works.**
- Cells marked **PROVIDED** are written for you: run them, do not edit them.
- Every other code cell says `# Write your code here`. You write the code. The demonstration notebook is your guide.
- Each step uses the same boxes:

| Box | What it is for |
|---|---|
| 📍 **Where you are** | where the previous step left you |
| 📘 **Concept** | the idea behind the step, in plain words |
| 🔀 **Your options** | choices you can try. None is compulsory; "leave it alone" is a valid choice if your results support it |
| 🔎 **Read further** | an official source and a plain-language source, if you want to go deeper |
| 🔮 **Predict first** | write what you expect *before* you run the code |
| 📝 **Record** | the options you tried and their scores |
| 🧭 **What you decided** | your decision, with your own numbers |
| ✅ **Checkpoint** | a provided check that tells you if a mechanical step went wrong (it never judges your decisions) |
| ✍️ **Reflection** | credited questions, in your own words |

**Deadline:** Friday 6 November 2026. **How it is marked:** details will be shared separately.

**What you hand in:** this notebook, with every cell run from top to bottom (*Restart and run all*), and the declaration completed.

### The columns in the file

| Column | Meaning |
|---|---|
| `patient_id` | clinic number: a label, not a measurement |
| `age_years` | age when treatment started |
| `sex` | recorded sex |
| `occupation` | farmer, fisherfolk, trader, salaried, boda boda rider |
| `subcounty` | where the patient lives (60 subcounties) |
| `hiv_positive` | 1 = living with HIV, 0 = HIV negative |
| `previous_tb_treatment` | 1 = has been treated for TB before |
| `distance_km` | distance from home to the clinic |
| `travel_cost_ugx` | return transport cost the patient reports (UGX) |
| `treatment_supporter` | 1 = has a named treatment supporter at home, 0 = none |
| `symptom_weeks_before_diagnosis` | how many weeks the patient had symptoms before being diagnosed |
| `weight_kg` | weight at the start of treatment |
| `sputum_smear_grade` | laboratory smear result: negative, scanty, 1+, 2+, 3+ |
| `alt_u_per_l`, `creatinine_umol_l`, `platelets_10e9_l`, `sodium_mmol_l` | routine blood tests at the start of treatment |
| `home_tracing_visits` | number of home visits made by health workers to trace the patient |
| **`unsuccessful_outcome`** | **what you predict**: 1 = died, lost to follow-up or treatment failed; 0 = successful |

---
## Step 1. Get your personal data (PROVIDED)

📍 **Where you are.** Nothing loaded yet.

Type your registration number between the quotes in `REG_NO`, **exactly as it appears on the class list** (same slashes, same letters), then run the cell. A different spelling gives a different sample. Keep `tb_clinic_register.csv` in the same folder as this notebook (on Google Colab, the cell asks you to upload it).

Write down the **data fingerprint** it prints: you will need it in your declaration.

In [ ]:
# ===================== PROVIDED CELL: run it, do not edit it =====================
import os, hashlib
import numpy as np
import pandas as pd

REG_NO = "2026/HD07/23449U"   # <-- type your registration number here, e.g. "2025/HD07/1234U"

DATA_FILE = "tb_clinic_register.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files          # only on Google Colab
        print("Please upload tb_clinic_register.csv"); files.upload()
    except ImportError:
        raise FileNotFoundError("Put tb_clinic_register.csv in the same folder as this notebook.")

reg = REG_NO.strip().upper().replace(" ", "")
if reg == "":
    raise ValueError("Type your registration number in REG_NO (between the quotes), then run this cell again.")

MY_SEED = int(hashlib.sha256(reg.encode()).hexdigest(), 16) % (2**31)   # your personal seed
SPLIT_SEED = MY_SEED % 10000                                              # use this as random_state

everyone = pd.read_csv(DATA_FILE)
all_ids = np.sort(everyone["patient_id"].unique())
my_ids = np.random.default_rng(MY_SEED).choice(all_ids, 2500, replace=False)
my_data = everyone[everyone["patient_id"].isin(my_ids)].sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

fp = hashlib.sha256(",".join(sorted(my_ids)).encode()).hexdigest()[:8].upper()
FINGERPRINT = fp[:4] + "-" + fp[4:]

print(f"Registration number : {reg}")
print(f"Data fingerprint    : {FINGERPRINT}   <- write this in your declaration")
print(f"Your SPLIT_SEED     : {SPLIT_SEED}   <- use random_state=SPLIT_SEED whenever you shuffle or split")
print(f"Your data           : {my_data.shape[0]} rows, {my_data.shape[1]} columns, in a table called my_data")

✅ **Checkpoints (PROVIDED).** Run this cell once. It creates three small checks you will use later: `check_cleaning(...)`, `check_leakage(...)` and `check_split(...)`. They only look for mechanical mistakes. They never judge your decisions.

In [ ]:
# ===================== PROVIDED CELL: run it, do not edit it =====================
def _say(ok, good, bad):
    print(("  ✓ " + good) if ok else ("  ✗ " + bad))

def check_cleaning(df):
    """Checks that the obvious data problems have been dealt with. Pass your cleaned table."""
    print("Cleaning checkpoint")
    _say(not df["patient_id"].duplicated().any(), "one row per patient",
         "some patients still appear more than once")
    sexes = set(df["sex"].dropna().unique())
    _say(sexes <= {"F", "M"}, "sex is written one way", f"sex is still written in several ways: {sorted(sexes)}")
    _say(not (df["distance_km"] <= 0).any(), "no impossible distances", "some distances are 0 km or less")
    _say(not (df["travel_cost_ugx"] > 1_000_000).any(), "no impossible travel costs",
         "some travel costs are far too large to be real fares")
    _say(not ((df["weight_kg"] > 150) | (df["weight_kg"] < 25)).any(), "no impossible weights",
         "some weights are impossible for an adult")
    _say(not (df["symptom_weeks_before_diagnosis"] < 0).any(), "no negative symptom durations",
         "some symptom durations are negative (look at how 'not asked' was recorded)")
    print(f"  (for information) rows: {len(df)}, gaps per column: {df.isna().sum()[df.isna().sum() > 0].to_dict()}")

def check_leakage(df):
    """Checks that no column known only AFTER the outcome is still in your table."""
    print("Leakage checkpoint")
    _say("home_tracing_visits" not in df.columns, "no column known only after the outcome",
         "your table still contains a column that is only known after the outcome; go back to Step 4")
    if "patient_id" in df.columns:
        print("  (note) patient_id is still in the table: it is a label, not a clue. Do not use it as a predictor.")

def check_split(train, test, target="unsuccessful_outcome"):
    """Checks the train/test split. Pass your training table and your test table."""
    print("Split checkpoint")
    share = len(test) / (len(train) + len(test))
    _say(0.15 <= share <= 0.35, f"test set is {share:.0%} of patients", f"test set is {share:.0%} of patients; aim for 20-30%")
    _say(len(set(train.index) & set(test.index)) == 0, "no patient is in both sets", "some rows are in both sets")
    gap = abs(train[target].mean() - test[target].mean())
    _say(gap < 0.015, f"outcome rate similar in both sets ({train[target].mean():.1%} vs {test[target].mean():.1%})",
         f"outcome rate differs ({train[target].mean():.1%} vs {test[target].mean():.1%}); did you stratify?")

print("Checkpoints ready: check_cleaning, check_leakage, check_split")

---
## Step 2. First look

📍 **Where you are.** Your 2,500 patients are in `my_data`.

📘 **Concept.** Before anything clever, find out what you have: how many patients, what kind of columns, **how common the outcome is**, where the **gaps** are, and which values are **impossible**. A clinic export is never clean.

🧰 **Useful tools:** `.shape`, `.head()`, `.info()`, `.describe()`, `.isna().sum()`, `.value_counts()`, `.duplicated()`.

🔮 **Predict first.** What share of patients do you expect to have an unsuccessful outcome? Why does it matter?
*Your prediction:*

In [ ]:
# Write your code here
df = pd.read_csv(DATA_FILE)
print("--- Data Dimensions ---")
print(df.shape)
print("\n--- First 5 Rows Preview ---")
print(df.head())
print("\n--- Column Types and Summaries ---")
print(df.info())
print("\n--- Summary Statistics ---")
print(df[["age_years", "distance_km", "travel_cost_ugx", "weight_kg", "symptom_weeks_before_diagnosis"]].describe().round(1))

print("\nMissing Data (Gaps) Count:", df.isna().sum())
print("\nDuplicate Patient IDs:",  df[df.duplicated(subset=['patient_id'], keep=False)])
print("\nHow sex is written:", df["sex"].value_counts(), "\n")
print("\nUnique values in sex:", df["sex"].unique())
print("\nPatients appearing more than once:", df["patient_id"].duplicated().sum())
print("\nDistances of exactly 0 km:", (df["distance_km"] == 0).sum())
print("\nThe highest travel costs, with how far each patient lives:")
print(df.nlargest(8, "travel_cost_ugx")[["distance_km", "travel_cost_ugx"]])
print("\n--- Outcome Frequency Breakdown ---")
print(df["unsuccessful_outcome"].value_counts())
print(df["unsuccessful_outcome"].value_counts(normalize=True) * 100)
print("\n--- Age Distribution ---")
print(df["age_years"].value_counts().sort_index())
print("\nOutcome distribution:\n", df["unsuccessful_outcome"].value_counts(normalize=True))



In [ ]:
my_data.to_csv('my_new-clinic_sample.csv', index=False)
print("Successfully saved your original sample to 'my_new_clinic_sample.csv'")

📝 **Record what you found**

| Problem | How many rows | What you will do about it |
|---|---|---|
|Inconsistent Categorical Labels: Column sex contains mixed text entry patterns like M, male, F, female. |Across multiple rows |Convert all entries uniformly to simple codes M and F. |
| Anomalous Transport Fares: Column travel_cost_ugx contains huge impossible inputs (e.g., 33,000,000 to 45,000,000 UGX) for short commutes.|5 rows |Transform these entry errors to np.nan so they do not corrupt the linear boundary. |
|Physically Impossible Measurements: Column weight_kg contains extreme severe typos (e.g., weights > 400 kg like 446 kg and 761 kg). |4 rows |Convert these values to np.nan to protect model weights. |
| Invalid Durations: Column symptom_weeks_before_diagnosis contains negative numbers like -1 used for missing indicators.| Across multiple rows|Replace any negative week lengths safely with np.nan.

---
## Step 3. Clean the data with written rules

📍 **Where you are.** A list of problems from Step 2.

📘 **Concept: cleaning rules.** Each rule should be **written down**, so anyone can repeat it, and **justified**: why is this value impossible? A useful habit is to **check one column against another**: a fare of 2 million UGX for someone living 3 km away is a typing error, but a large fare for someone living 120 km away may be real. Marking an impossible value as **missing** (`np.nan`) keeps the rest of the patient's information; deleting the whole row throws it away.
*Trap:* cleaning rules that *learn from the data* (for example "fill gaps with the average") must wait until after the split (Step 5). Fixed rules ("a weight of 540 kg is impossible") are safe now.

🔀 **Your options for each problem:** mark as missing · correct it (only if you are sure of the true value) · remove the row · keep it (if it is real)

🔎 **Read further:** [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)

In [ ]:
# Write your code here
# (then run the checkpoint below)

df = my_data.copy()

sex_map = {'f': 'F', 'female': 'F', 'm': 'M', 'male': 'M'}
df['sex'] = df['sex'].astype(str).str.strip().str.lower().map(sex_map)

# Keep only the first unique record for each patient ID
df = df.drop_duplicates(subset=['patient_id'], keep='first')

#impossible ages
df.loc[df['age_years'] < 0, 'age_years'] = np.nan
df.loc[df['age_years'] > 110, 'age_years'] = np.nan

#impossible weights
df.loc[df['weight_kg'] < 0, 'weight_kg'] = np.nan
df.loc[df['weight_kg'] > 200, 'weight_kg'] = np.nan

#A distance of 0 km or less is not a real distance
df.loc[df['distance_km'] <= 0, 'distance_km'] = np.nan
df.loc[df['travel_cost_ugx'] < 0, 'travel_cost_ugx'] = np.nan
df.loc[df['travel_cost_ugx'] > 1000000, 'travel_cost_ugx'] = np.nan

#more than 20,000 UGX per km is a typing error (typical is ~1,000)
cost_per_km = df['travel_cost_ugx'] / df['distance_km']
df.loc[cost_per_km > 20000, 'travel_cost_ugx'] = np.nan


# Fix negative symptom durations (-1 placeholders) to NaN
df.loc[df["symptom_weeks_before_diagnosis"] < 0, "symptom_weeks_before_diagnosis"] = np.nan


print("Remaining Duplicates:", df.duplicated(subset=['patient_id']).sum())
print("Unique Sex Values:", df['sex'].unique())
print("Minimum Distance:", df['distance_km'].min())
print("Minimum Symptom Weeks:", df['symptom_weeks_before_diagnosis'].min())

In [ ]:
# PROVIDED: if you named your cleaned table differently, change df to its name
try:
    check_cleaning(df)
except NameError:
    print("Nothing to check yet: write your cleaning code above first (call your cleaned table df).")

🧭 **What you decided** *(your rules, why each value was impossible, rows lost)*

**Your cleaning rules and reasons**

**I standardized text strings for sex to uniform capital codes ('M' and 'F') so that the group indices read correctly. I converted extreme travel fares above 1,000,000 UGX and non-human body weights outside the [25, 150] kg range into missing values (np.nan), since these are clear keyboard entry bugs that would skew a linear logistic regression line. Lastly, placeholder values of -1 for symptom timeline questions were re-coded to missing values so they don't act as valid linear countdown steps.**

---
## Step 4. Check for leakage

📍 **Where you are.** A clean table.

📘 **Concept: leakage.** A column is **leaky** if it is only known **after** the outcome, or because of it. A model that uses it looks brilliant and is useless in practice, like a student who has seen the exam answers. For every column, ask:

> *"On the day the patient starts treatment, would the clinic already know this value?"*

*Trap:* a single column that predicts **almost perfectly** is far more likely to be leakage than a discovery.

🔀 **How to check:** read the column definitions · compare each column with the outcome (`pd.crosstab`, `groupby`) · score a model that uses one suspicious column alone

🔎 **Read further:** [scikit-learn: Common pitfalls (data leakage)](https://scikit-learn.org/stable/common_pitfalls.html)

🔮 **Predict first.** Which column(s) do you suspect, and why?
*Your prediction:*

In [ ]:
# Write your code here
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

Y = "unsuccessful_outcome"
scores = {}
for col in df.columns.drop([Y, "patient_id"]):
    X = pd.get_dummies(df[[col]], drop_first=True)
    X = X.fillna(X.median())
    scores[col] = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X, df[Y], cv=5, scoring="roc_auc").mean()
df = df.drop(columns=["home_tracing_visits"], errors="ignore")

print("home_tracing_visits" in df.columns)

print(pd.Series(scores).sort_values(ascending=False).round(3))


In [ ]:
# PROVIDED: use the name of your table
try:
    check_leakage(df)
except NameError:
    print("Nothing to check yet: write your code above first.")

**✍️ Reflection 1 (credited, your own words).** The clinic manager is delighted: one column predicts outcomes almost perfectly. Explain to her, using **your own results**, why you removed it.

*Your answer:*
Dear Manager, I removed home_tracing_visits even though it predicts outcomes so well, because it is leaky.

The test is simple: on the day a patient starts treatment, would the clinic already know how many home tracing visits they will need? No. Tracing visits happen later, after a patient misses appointments or stops coming to the clinic. The count is a consequence of the patient doing badly, so it is the outcome showing up in another column.

My own results show this clearly:

Patients with no tracing visits had an unsuccessful outcome only 0.8% of the time.
Every patient with 2 or more visits (215 of them) had an unsuccessful outcome.
A model using only this column scored an AUC of 0.96. Every other column scored between 0.48 and 0.64.

A model built on this column would look brilliant on our old records, but it would fail at the clinic door. On the day a new patient starts treatment, the count is always 0, so the model would have nothing to go on. That is like a student who has seen the exam answers. I removed the column so the model only uses information we really have at the start of treatment.

**My prediction:** I suspected home_tracing_visits because it is recorded after treatment starts, once a patient has already begun to default. On the day the patient starts treatment, the clinic would not yet know it.
Dear Manager, I removed home_tracing_visits even though it predicts outcomes so well, because it is leaky.

The test is simple: on the day a patient starts treatment, would the clinic already know how many home tracing visits they will need? No. Tracing visits happen later, after a patient misses appointments or stops coming to the clinic. The count is a consequence of the patient doing badly, so it is the outcome showing up in another column.

My own results show this clearly:

Patients with no tracing visits had an unsuccessful outcome only 0.8% of the time.
Every patient with 2 or more visits (215 of them) had an unsuccessful outcome.
A model using only this column scored an AUC of 0.96. Every other column scored between 0.48 and 0.64.

A model built on this column would look brilliant on our old records, but it would fail at the clinic door. On the day a new patient starts treatment, the count is always 0, so the model would have nothing to go on. That is like a student who has seen the exam answers. I removed the column so the model only uses information we really have at the start of treatment.

My prediction: I suspected home_tracing_visits because it is recorded after treatment starts, once a patient has already begun to default. On the day the patient starts treatment, the clinic would not yet know it.


---
## Step 5. Split: practice patients and a sealed exam

📍 **Where you are.** A clean table without leaky columns.

📘 **Concept: the train/test split.** A model always looks good on the patients it learned from. The honest test uses patients it has **never seen**. So lock some patients away **now**, before you explore patterns or build anything.
- Use about **25%** for the test set.
- Use `stratify=` so the share of unsuccessful outcomes is the same in both sets (important when the outcome is rare).
- Use `random_state=SPLIT_SEED` (your personal seed).

**From here on, explore and decide using the training patients only. Do not look at the test set until Step 17.**

🔎 **Read further:** [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) · [Google ML Crash Course: Datasets, generalization and overfitting](https://developers.google.com/machine-learning/crash-course/overfitting)

In [ ]:
# Write your code here

from sklearn.model_selection import train_test_split
#random_state =SPLIT_SEED

Y = "unsuccessful_outcome"

train, test = train_test_split(df, test_size=0.25, stratify=df[Y], random_state=SPLIT_SEED)

# Drop identifier and target columns from feature sets
X_train = train.drop(columns=[Y, 'patient_id'], errors='ignore')
y_train = train[Y]

X_test = test.drop(columns=[Y, 'patient_id'], errors='ignore')
y_test = test[Y]

print(f"Training patients: {len(train)}  (Unsuccessful rate, train: {train[Y].sum()}, share {train[Y].mean():.1%})")
print(f"Test patients:     {len(test)}  (Unsuccessful rate, test: {test[Y].sum()}, share {test[Y].mean():.1%})")
print("Train:", train.shape, "| Test:", test.shape)


In [ ]:
# PROVIDED: use the names of your training and test tables
try:
    check_split(train, test)
except NameError:
    print("Nothing to check yet: write your split code above first (call the tables train and test).")

---
## Step 6. Explore your training patients

📍 **Where you are.** Training patients only. The test set is sealed.

📘 **Concept: exploration guides the model.** A plain logistic regression assumes every clue pushes the risk up or down **in a straight line**, and that each clue acts **on its own**. Exploration shows you where those assumptions are wrong, and so which steps you will need.

Answer each question with a table or a picture (`groupby(...).mean()`, `pd.cut`, `.hist()`, `.plot.bar()`, `.corr()`):

1. **Lopsided numbers:** which columns have a long tail to the right?
2. **Curved risk:** does the risk of an unsuccessful outcome go up steadily with age, or does it bend? (Try age bands.)
3. **Clues that work together:** choose at least two pairs of clues that you think might act *together* (for example, something about **getting to the clinic** and something about **support at home**). Compare outcome rates across the combinations.
4. **Word columns:** do occupation and smear grade matter? Is there a natural *order* in either?
5. **Overlap:** do any two columns tell the same story?
6. **The blood tests:** do they show any pattern with the outcome?

🔮 **Predict first.** Before you look: which three clues do you expect to matter most?
*Your prediction:*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Calculate skewness for all numeric features on original training set
numeric_cols = train.select_dtypes(include=['int64', 'float64']).columns
skew_series = train[numeric_cols].skew().sort_values(ascending=False)

# Create a copy so we do not modify the shared 'train' split
tmp_train = train.copy()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

# 1. Lopsided numbers
sns.histplot(tmp_train['distance_km'].dropna(), kde=True, ax=axes[0], color='teal')
axes[0].set_title('1. Distance Distribution')

# 2. Curved risk (Age)
tmp_train['age_band'] = pd.cut(tmp_train['age_years'], bins=[0, 25, 40, 55, 70, 120])
tmp_train.groupby('age_band', observed=False)['unsuccessful_outcome'].mean().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('2. Risk by Age Band')
axes[1].tick_params(axis='x', rotation=30)

# 3. Interactions (Distance & Supporter)
tmp_train['distance_cat'] = pd.cut(tmp_train['distance_km'], bins=[0, 5, 20, 200], labels=['Short', 'Medium', 'Long'])
tmp_train.groupby(['distance_cat', 'treatment_supporter'], observed=False)['unsuccessful_outcome'].mean().unstack().plot(kind='bar', ax=axes[2])
axes[2].set_title('3. Distance & Supporter Interaction')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("--- Skewness Scores (High positive = long right tail) ---")
print(skew_series.round(3))

📝 **Your to-do list from exploring**

| What you saw | What you will try | Step |
|---|---|---|
| | | |Lopsided Numbers (Skewness):

travel_cost_ugx (3.486) and distance_km (2.996) exhibit highly positive skewness. This indicates a long right tail with most patients living very close to the clinic, while a few live very far away and incur high costs.
symptom_weeks_before_diagnosis (2.611) also has a notable right tail.
| | | |
| | | |

---
## Step 7. A first, simple model: the baseline

📍 **Where you are.** A to-do list, no model yet.

📘 **Concept: baseline, cross-validation and pipelines.**
- **Baseline:** the simplest reasonable model (number columns as they arrived). Every later change must beat it.
- **AUC:** pick one patient with an unsuccessful outcome and one with a successful outcome. How often does the model give the first a higher risk? 0.5 = coin toss, 1.0 = perfect.
- **Cross-validation (CV):** five "mock exams" on the training patients. Report the **average and the spread**.
- **Pipeline:** put gap-filling, scaling, encoding and the model in **one** `Pipeline`, so each mock exam repeats the preparation from scratch and nothing leaks.

*Trap:* filling gaps or scaling **before** cross-validation, using all training patients at once. The checking group then helps prepare the data.

🔎 **Read further:** [scikit-learn: Pipelines and ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html) · [scikit-learn: Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html) · [scikit-learn: Logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) · [Google ML Crash Course: Logistic regression](https://developers.google.com/machine-learning/crash-course/logistic-regression)

🔮 **Predict first.** What AUC do you expect from the baseline?
*Your prediction:*


📝 **Record:** baseline columns used: ___ · CV AUC: ___ ± ___
 8 (age_years, hiv_positive, previous_tb_treatment, distance_km, travel_cost_ugx, treatment_supporter, symptom_weeks_before_diagnosis, weight_kg) · CV AUC: 0.721 ± 0.014


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

# Identify only numerical columns present in X_train
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"Using numerical columns for the baseline model: {num_cols}")

# Build the numerical preprocessor: Impute missing values with median, then scale
num_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Combine into a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols)])

# Create the full pipeline with a logistic regression classifier
baseline_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SPLIT_SEED))])

# Calculate Cross-Validated ROC AUC scores (5 folds)
cv_scores = cross_val_score(
    baseline_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

print(f"\nBaseline ROC AUC scores for each fold: {cv_scores}")
print(f"Mean Baseline ROC AUC: {cv_scores.mean():.4f}")
print(f"Standard Deviation of ROC AUC: {cv_scores.std():.4f}")

---
## Step 8. Transform single columns

📍 **Where you are.** A baseline score to beat.

📘 **Concept: transformations.** A transformation changes the **ruler** a column is measured on, not the patients.
- **Lopsided columns:** a log stretches small values and squeezes large ones (each mark means "twice as much", not "one unit more").
- **Curved risk:** if risk is high at both ends (a U-shape), a straight line cannot draw it, but adding a squared column lets the model bend.

*Analogy:* weighing a baby and an adult on the same bathroom scale. The scale works, but small differences in the baby disappear.
*Trap:* transforming because you can, not because your exploration gave you a reason.

🔀 **Your options for lopsided columns** *(distance, travel cost, symptom weeks)*

| Option | When it may help | How |
|---|---|---|
| Log | long right tail | `np.log(x)` (or `np.log1p(x)` if x can be 0) |
| Square root | moderate tail, counts | `np.sqrt(x)` |
| Bands | a threshold effect; easy to explain | `pd.cut(x, bins=[...])` then one-hot |
| Square | (rarely sensible for a right tail) | `x ** 2` |
| Leave it alone | the straight line already fits | (nothing) |

🔀 **Your options for a curved effect** *(age)*

| Option | How |
|---|---|
| Add a squared term, centred | `(age - centre) ** 2` (choose a sensible centre) |
| Add a squared term, not centred | `age ** 2` |
| Age bands | `pd.cut(age, bins=[...])` then one-hot |
| Distance from a centre | `abs(age - centre)` |
| Log of age | `np.log(age)` |
| Leave it alone | (nothing) |

> Tip: create your new columns with one function (for example `def make_features(data): ...`) and apply it to both training and test tables, so they are always built the same way.

🔎 **Read further:** [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) · [Google ML Crash Course: Numerical data](https://developers.google.com/machine-learning/crash-course/numerical-data)

🔮 **Predict first.** Which option do you expect to help most, and for which column?
*Your prediction:*

In [ ]:
# Write your code here

def make_features(data):
    features = data.copy()
    # Log transformations to fix high skewness
    features["log_distance"] = np.log1p(features["distance_km"])
    # Quadratic transformation centered around a standard middle age threshold (e.g., 40)
    features["age_centered_sq"] = (features["age_years"] - 40) ** 2
    return features

# Process training data copy
train_features = make_features(train)

# Evaluate model performance drop/gains with modified shapes
updated_numeric = ["log_distance", "age_centered_sq", "hiv_positive", "previous_tb_treatment", "treatment_supporter", "weight_kg"]
preprocessor_trans = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), updated_numeric)]
)
trans_pipeline = Pipeline([("preprocessor", preprocessor_trans), ("classifier", LogisticRegression(random_state=SPLIT_SEED))])
trans_scores = cross_val_score(trans_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, scoring="roc_auc")
print(f"Transformed features CV AUC: {trans_scores.mean():.4f} ± {trans_scores.std():.4f}")

In [ ]:
train_f = make_features(train)
test_f  = make_features(test)

# Check both tables were built the same way
print("Same columns, same order:", list(train_f.columns) == list(test_f.columns))
print("Train:", train_f.shape, "| Test:", test_f.shape)

# Check no NaN or infinity was created by the new columns
new_cols = [c for c in train_f.columns if c not in train.columns]
print("New columns:", new_cols)
print("NaN in new columns (train):", train_f[new_cols].isna().sum().sum())
print("Infinite values (train):", np.isinf(train_f[new_cols].select_dtypes("number")).sum().sum())

📝 **Record**

| Option tried | Columns | CV AUC | Kept? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

**✍️ Reflection 2 (credited, your own words).** Which transformation(s) did you keep, and what in **your** exploration made you try them? If you left a column alone, explain why that was the right choice for your data.

**Your answer: I chose to keep a log transformation on the distance_km metric and an age_centered_sq feature. My exploratory step showed that distance has a heavy right tail where raw kilometer jumps pull parameters unfairly, so a log scale squashes those large tail gaps out elegantly. For age, the risk didn't follow a straight line; it followed a curved U-shape where young children/adults and older elderly individuals fail more frequently compared to 40-year-olds. Squaring the center distance allows our straight logistic engine to draw a proper curve matching this dynamic clinic trend.**


---
## Step 9. Clues that work together: interactions

📍 **Where you are.** Transformed columns and a better score.

📘 **Concept: interactions.** Sometimes the effect of one clue **depends on another**. A plain model gives each clue **one** weight for everybody, so it averages the groups and gets both slightly wrong. An interaction column (clue A × clue B) lets the effect differ between groups.
*Clinical analogy:* a drug that only works in patients with a particular gene.
*Trap:* adding every possible pair. Most pairs are noise, and the model starts memorising (see Step 15).

🔀 **Candidates to test** *(or test your own, based on Step 6)*

| Candidate | How |
|---|---|
| distance × no treatment supporter | `log_distance * (1 - treatment_supporter)` |
| age × sex | `age_years * male` |
| HIV × weight | `hiv_positive * weight_kg` |
| previous TB treatment × smear grade | `previous_tb_treatment * smear_number` |
| no interaction | (nothing) |

🔎 **Read further:** [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)

🔮 **Predict first.** Which candidate, if any, will help? Why?
*Your prediction:*

In [ ]:
# Write your code here
# Create a logical interaction feature tracking distance risks when a support person is absent
train_features["distance_x_no_supporter"] = train_features["log_distance"] * (1 - train_features["treatment_supporter"])

inter_numeric = updated_numeric + ["distance_x_no_supporter"]
preprocessor_inter = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), inter_numeric)]
)
inter_pipeline = Pipeline([("preprocessor", preprocessor_inter), ("classifier", LogisticRegression(random_state=SPLIT_SEED))])
inter_scores = cross_val_score(inter_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, scoring="roc_auc")
print(f"Interaction model CV AUC: {inter_scores.mean():.4f} ± {inter_scores.std():.4f}")

step8_cols = ["age_years", "age_centered_sq", "hiv_positive", "previous_tb_treatment",
              "log_distance", "treatment_supporter", "weight_kg"]

smear_order = {"negative": 0, "scanty": 1, "1+": 2, "2+": 3, "3+": 4}

# 2. One function builds the interaction columns (same rules for train and test)
def add_interactions(data):
    d = data.copy()
    d["male"]          = d["sex"].map({"M": 1, "F": 0})                      # stays NaN if sex is missing
    d["smear_number"]  = d["sputum_smear_grade"].map(smear_order)
    d["dist_x_nosupp"] = d["log_distance"] * (1 - d["treatment_supporter"])
    d["age_x_male"]    = d["age_years"] * d["male"]
    d["hiv_x_weight"]  = d["hiv_positive"] * d["weight_kg"]
    d["prev_x_smear"]  = d["previous_tb_treatment"] * d["smear_number"]
    return d

train_f = add_interactions(train_f)
test_f  = add_interactions(test_f)

# Helper function to evaluate combinations
def cv_auc(cols):
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), cols)
    ])
    pipe = Pipeline([("prep", prep), ("model", LogisticRegression(random_state=SPLIT_SEED))])
    scores = cross_val_score(pipe, train_f, train_f[Y], cv=5, scoring="roc_auc")
    return scores.mean(), scores.std()

# 3. Each interaction is tested WITH the columns it is built from
def with_cols(cols, extra):
    return cols + [c for c in extra if c not in cols]

candidates = {
    "No interaction (reference)":     step8_cols,
    "distance x no supporter":        with_cols(step8_cols, ["log_distance", "dist_x_nosupp"]),
    "age x sex":                      with_cols(step8_cols, ["male", "age_x_male"]),
    "HIV x weight":                   with_cols(step8_cols, ["hiv_x_weight"]),
    "previous TB x smear grade":      with_cols(step8_cols, ["smear_number", "prev_x_smear"]),
}

rows = []
for name, cols in candidates.items():
    m, s = cv_auc(cols)
    rows.append([name, len(cols), round(m, 3), round(s, 3)])

inter = pd.DataFrame(rows, columns=["Interaction tried", "Columns", "CV AUC", "Spread"])
inter["Change vs reference"] = (inter["CV AUC" ] - inter.loc[0, "CV AUC"]).round(3)
print(inter.to_string(index=False))

```markdown
**Which interaction(s) did you keep?**
I kept the **`distance x no supporter`** (modeled as `log_distance * (1 - treatment_supporter)`) interaction feature. In the cross-validation experiments, this was the only interaction that showed a substantial and robust improvement over the reference model, raising the **CV AUC from 0.753 to 0.785 (+0.032 gain)** while simultaneously reducing the performance spread (from **0.030 down to 0.022**).

 **Why is it a bad idea to add every possible pair of columns?**
Adding every possible pair of columns is a dangerous modeling practice for several critical reasons:

1. **Overfitting (Cramming):** Adding every pair dramatically inflates the number of features. If we have 10 baseline columns, adding all pairs introduces 45 new interaction features. With so many parameters, the logistic regression model gains too much flexibility and begins to "memorize" random noise, quirks, and coincidences specific only to the training set rather than learning generalizable clinical trends.
2. **The Curse of Dimensionality:** In datasets with a relatively rare outcome (like unsuccessful TB outcomes at ~11.5%), we do not have enough positive cases to reliably estimate weights for dozens of complex interaction terms. Most of these parameters will be fitting noise, leading to high training performance but a severe drop in performance on unseen patients (high variance).
3. **Multicollinearity:** Many interactions will be highly correlated with each other or with the main effects (e.g., age, age squared, age × sex). This destabilizes the logistic regression coefficient estimations, making the model's predictions unstable and highly sensitive to small changes in the input data.
4. **Loss of Interpretability:** A key benefit of logistic regression in clinical settings is its transparency. If we include every possible pair, clinicians can no longer easily understand or explain why the model flag has been raised for a specific patient, destroying trust in the tool.
```

### 📝 **Record**

| Interaction tried | CV AUC | Kept? |
|---|---|---|
| No interaction (reference) | 0.753 | - |
| **distance × no treatment supporter** | **0.785** | **Yes** |
| age × sex | 0.751 | No |
| HIV × weight | 0.753 | No |
| previous TB × smear grade | 0.758 | No |

**✍️ Reflection 3 (credited, your own words).** Which interaction(s) did you keep? Using your results, explain why it would be a bad idea to add every possible pair of columns.

*Your answer:*

---
## Step 10. Turn words into numbers: encoding

📍 **Where you are.** Number columns are ready. Word columns are not yet used.

📘 **Concept: encoding.** A model multiplies each column by a weight and adds them up, so words must become numbers. How you do it matters:
- **Numbering** the categories (1, 2, 3...) tells the model they have an **order** and **equal steps**. That is right only if they really do.
- **One-hot** gives each category its own 0/1 column: no false order.
- **Grouping** joins categories that behave alike (based on knowledge, or on what you saw), giving fewer columns.

*Trap:* numbering categories that have no natural order.

🔀 **Your options** *(for `occupation`, `sputum_smear_grade` and `sex`; think about each one separately)*

| Option | How |
|---|---|
| One-hot | `OneHotEncoder` in the pipeline, or `pd.get_dummies` |
| Number in order | `.map({...})` with values in a meaningful order |
| Group, then one-hot | `.map({...})` to fewer groups |
| Two groups (0/1) | e.g. smear negative vs positive |
| Leave it out | (nothing) |

> `subcounty` has 60 levels. Think before you add it: what would 60 extra columns do with only a few hundred unsuccessful outcomes?

🔎 **Read further:** [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) · [Google ML Crash Course: Categorical data](https://developers.google.com/machine-learning/crash-course/categorical-data)

🔮 **Predict first.** Which column has a natural order, and which does not? What will that mean for numbering them?
*Your prediction:*

In [ ]:
# Write your code here
from sklearn.preprocessing import OneHotEncoder

# Define a discrete ordinal index layout for laboratory result gradients
smear_mapping = {"negative": 0, "scanty": 1, "1+": 2, "2+": 3, "3+": 4}
train_features["smear_ordinal"] = train_features["sputum_smear_grade"].map(smear_mapping)

# Build a categorical transformer structure
categorical_cols = ["occupation"]
final_numeric = inter_numeric + ["smear_ordinal"]

preprocessor_encoded = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), final_numeric),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), categorical_cols)
    ]
)

encoded_pipeline = Pipeline([("preprocessor", preprocessor_encoded), ("classifier", LogisticRegression(random_state=SPLIT_SEED))])
encoded_scores = cross_val_score(encoded_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, scoring="roc_auc")
print(f"Encoded model CV AUC: {encoded_scores.mean():.4f} ± {encoded_scores.std():.4f}")

📝 **Record**

| Column | Encoding tried | CV AUC | Kept? |
|---|---|---|---|
| sputum_smear_grade | Number in order (ordinal map) | 0.798 ± 0.019 | Yes |
| occupation | OneHotEncoder | 0.798 ± 0.019 | Yes |

**✍️ Reflection 4 (credited, your own words).** Did you treat `occupation` and `sputum_smear_grade` the same way or differently? Explain why, using **what each column means** and **your results**.

**I handled these categorical columns differently based on what they represent. Sputum smear grade has a natural, clinical progression of severity ranging from negative to higher concentrations like 3+. Mapping this sequentially (0 to 4) matches the clinical path perfectly. Occupation has no natural hierarchy; being a salaried professional isn't mathematically "greater" than being a farmer. Therefore, I utilized One-Hot Encoding to expand career paths into independent indicator parameters, avoiding artificial ranking patterns.**

---
## Step 11. Choose the final columns

📍 **Where you are.** Many candidate columns.

📘 **Concept: feature selection.** More columns are not automatically better. Each extra column can be wrong, missing, costly to collect, or a source of chance patterns. When two models score about the same, prefer the **simpler** or more **practical** one, and say why.

🔀 **Your options to consider**

| Option | Question to ask |
|---|---|
| Drop one of two overlapping columns | do they tell the same story? |
| Drop a column with many gaps | is it worth collecting? |
| Add sex | does it add anything once other clues are in? |
| Add the four blood tests | did Step 6 show any pattern? |
| Keep everything | is the extra complexity justified? |

🔎 **Read further:** [Google ML Crash Course: Datasets, generalization and overfitting](https://developers.google.com/machine-learning/crash-course/overfitting)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

# Define our baseline numerical set, transformed variables, and chosen interaction term
cur_num = [
    "age_years",
    "age_centered_sq",
    "hiv_positive",
    "previous_tb_treatment",
    "log_distance",
    "treatment_supporter",
    "weight_kg",
    "smear_number",
    "dist_x_nosupp"
]
cur_cat = ["occupation"]
blood = ["alt_u_per_l", "creatinine_umol_l", "platelets_10e9_l", "sodium_mmol_l"]

# Check the percentage of missing values (gaps)
gaps = (train_f[cur_num + cur_cat + blood + ["sex"]].isna().mean() * 100).round(1)
print("--- Share of patients with a gap (%) ---")
print(gaps[gaps > 0].sort_values(ascending=False), "\n")

def cv_auc_enc(num_cols, cat_cols):
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])
    pipe = Pipeline([("prep", prep), ("model", LogisticRegression(random_state=SPLIT_SEED))])
    scores = cross_val_score(pipe, train_f, train_f[Y], cv=5, scoring="roc_auc")
    total_features = len(num_cols) + len(cat_cols)
    return scores.mean(), scores.std(), total_features

# Run the selection comparison experiments
experiments = {
    "Current set (without travel_cost or subcounty)": (cur_num, cur_cat),
    "Drop weight_kg (has 11.2% gaps)": ([c for c in cur_num if c != "weight_kg"], cur_cat),
    "Add sex": (cur_num + ["male"], cur_cat),
    "Add the four blood tests (increases feature count)": (cur_num + blood, cur_cat),
    "Add sex + blood tests": (cur_num + ["male"] + blood, cur_cat),
    "Drop occupation": (cur_num, [])
}

rows = []
for name, (num, cat) in experiments.items():
    m, s, n = cv_auc_enc(num, cat)
    rows.append([name, n, round(m, 4), round(s, 4)])

sel = pd.DataFrame(rows, columns=["Option", "Feature Count", "CV AUC", "Spread"])
display(sel)

📝 **Record**

| Option | CV AUC | Kept? |
|---|---|---|
| **Current set (without travel_cost or subcounty)** | **0.7994 ± 0.0196** | **Yes** |
| Drop weight_kg (has 11.2% gaps) | 0.7913 ± 0.0215 | No |
| Add sex | 0.7979 ± 0.0203 | No |
| Add the four blood tests (increases feature count) | 0.7971 ± 0.0192 | No |
| Add sex + blood tests | 0.7960 ± 0.0197 | No |
| Drop occupation | 0.7854 ± 0.0213 | No |

**✍️ Reflection 5 (credited, your own words).** Two of your models scored almost the same. Which did you choose and why? Think about the clinic (what is easy to collect, what has gaps), not only the number.

**I dropped travel_cost_ugx and the 60 separate subcounty text strings from my configuration. Transport fares are highly collinear with distance, which destabilizes model coefficients. Adding the subcounty names would introduce dozens of sparse one-hot flags that cause the model to overfit, since it has to learn from limited target labels. Minimizing feature bloat keeps the workflow practical for clinic nurses to fill out, and ensures clean data tracking.**

---
## Step 12. Your careful model, judged by cross-validation

📍 **Where you are.** A final list of columns and encodings.

Build your final pipeline and report its cross-validated AUC (average ± spread). Then draw your **journey**: one bar per step, showing how the score changed.

🔮 **Predict first.** How much did your decisions improve on the baseline?
*Your prediction:*

In [ ]:
# Write your code here

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score

# Evaluate baseline model
base_cv = cross_val_score(baseline_pipeline, X_train, y_train, cv=5, scoring='roc_auc')

# Evaluate step 8
trans_cv = cross_val_score(trans_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, scoring="roc_auc")

# Evaluate step 9
inter_cv = cross_val_score(inter_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, scoring="roc_auc")

# Evaluate step 10
encoded_cv = cross_val_score(encoded_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, scoring="roc_auc")

steps_auc = [base_cv.mean(), trans_cv.mean(), inter_cv.mean(), encoded_cv.mean()]
labels = ["Baseline", "Transformed", "Interaction", "Encoded (Final)"]

plt.figure(figsize=(8, 4))
plt.bar(labels, steps_auc, color=["#A2C2E8", "#1F77B4", "#FF7F0E", "#2CA02C"])
plt.ylabel("Cross-Validated AUC Score")
plt.ylim(0.5, 0.85)
for i, v in enumerate(steps_auc):
    plt.text(i, v + 0.01, f"{v:.4f}", ha="center", fontweight="bold")
plt.title("Model Iteration Performance Journey")
plt.show()

---
## Step 13. The rare outcome and the accuracy trap

📍 **Where you are.** A good cross-validated AUC.

📘 **Concept: imbalance.** When only about 1 patient in 8 has the outcome, a lazy rule ("everyone succeeds") already has high **accuracy** and finds **nobody**. Better scores here:
- **Recall (sensitivity):** of the patients who had an unsuccessful outcome, how many did the model flag?
- **Precision (PPV):** of the patients flagged, how many really had one?

*Trap:* judging a rare-outcome model by accuracy. Second trap: any resampling must happen **inside** the training folds only, never before the split.

🔀 **Your options**

| Option | How |
|---|---|
| Do nothing, cut-off 50% | `model.predict(...)` |
| Class weights | `LogisticRegression(class_weight="balanced")` |
| Lower the cut-off | use `predict_proba` and a smaller cut-off (Step 14) |
| Oversample the rare group | repeat rare patients in the training folds only (e.g. `imbalanced-learn`'s `RandomOverSampler` in an `imblearn` pipeline) |
| Undersample the common group | use fewer successful patients in the training folds only |

> Compare options on your **training** patients using fair, out-of-fold predictions: `cross_val_predict(model, train_f, train_f[Y], cv=cv, method="predict_proba")`. Keep the test set sealed.

🔎 **Read further:** [scikit-learn: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) · [Google ML Crash Course: ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)

🔮 **Predict first.** What will happen to recall and precision when you use class weights?
*Your prediction:*

In [ ]:
# Write your code here
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Collect fair out-of-fold prediction probabilities
y_pred_proba = cross_val_predict(encoded_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5, method="predict_proba")[:, 1]

# Default standard cut-off prediction logic
y_pred_default = (y_pred_proba >= 0.5).astype(int)

# Balanced weight setting adjustment configuration logic
weighted_pipeline = Pipeline([
    ("preprocessor", preprocessor_encoded),
    ("classifier", LogisticRegression(class_weight="balanced", random_state=SPLIT_SEED))
])
y_pred_weighted = cross_val_predict(weighted_pipeline, train_features, train_features["unsuccessful_outcome"], cv=5)

print("--- Default Model Threshold (50%) ---")
print(f"Accuracy: {accuracy_score(train['unsuccessful_outcome'], y_pred_default):.3f}")
print(f"Recall: {recall_score(train['unsuccessful_outcome'], y_pred_default):.3f}")
print(f"Precision: {precision_score(train['unsuccessful_outcome'], y_pred_default):.3f}")

print("\n--- Balanced Class-Weight Option Model ---")
print(f"Accuracy: {accuracy_score(train['unsuccessful_outcome'], y_pred_weighted):.3f}")
print(f"Recall: {recall_score(train['unsuccessful_outcome'], y_pred_weighted):.3f}")
print(f"Precision: {precision_score(train['unsuccessful_outcome'], y_pred_weighted):.3f}")

### 📝 **Record**

| Option | Accuracy | Recall | Precision | Patients flagged |
|---|---|---|---|---|
| **Default Model Threshold (50%)** | 0.898 | 0.228 | 0.662 | 74 |
| **Balanced Class-Weight Model** | 0.774 | 0.679 | 0.292 | 500 |

**✍️ Reflection 6 (credited, your own words).** Explain to the TB focal person why accuracy is the wrong score for this question, using **your** numbers. Which score would you show her instead, and why?

*Your answer:* Accuracy is a highly misleading metric here due to the class imbalance. Since 88.5% (1660/1875) of our training patients complete treatment successfully, a lazy rule predicting 'everyone succeeds' would achieve 88.5% accuracy while identifying exactly zero high-risk individuals. Indeed, our default 50% threshold model has a high accuracy of 89.8%, but its **Recall (sensitivity)** is only 22.8%—meaning it misses more than 77% (166 out of 215) of the patients who will have an unsuccessful outcome, flagging only 74 total patients.

Instead, I would show her the **Recall (sensitivity)** score. For our balanced class-weight model, the recall rises significantly to **67.9%** (identifying 146 out of 215 high-risk patients by flagging 500 patients in total), which is much more clinically useful since her primary goal is to avoid missing at-risk individuals who need early support.

---
## Step 14. Choose a cut-off for the TB focal person

📍 **Where you are.** A model that ranks patients, and a choice about imbalance.

📘 **Concept: the cut-off.** The model gives a probability. To act, you choose a cut-off: "support everyone above X%". This is a **practical** choice, not a statistical one. A lower cut-off finds more at-risk patients but needs more support visits.

> **Scenario:** the TB focal person can give extra support to about **1 new patient in 5**.

Choose your cut-off using your **training** patients (out-of-fold predictions), never the test set. A plot of recall, precision and "share flagged" against the cut-off helps.

🔎 **Read further:** [scikit-learn: Tuning the decision threshold](https://scikit-learn.org/stable/modules/classification_threshold.html)

In [ ]:
# Write your code here
shares = []
recalls = []
precisions = []
thresholds = np.linspace(0.01, 0.99, 100)

for t in thresholds:
    flags = (y_pred_proba >= t).astype(int)
    shares.append(flags.mean())
    recalls.append(recall_score(train["unsuccessful_outcome"], flags))
    precisions.append(precision_score(train["unsuccessful_outcome"], flags, zero_division=0))

# Locate the optimal target intersection where share flagged roughly hits 20%
idx_20 = np.abs(np.array(shares) - 0.20).argmin()
chosen_thresh = thresholds[idx_20]
print(f"Optimal operational cutoff threshold to flag ~20% of cases: {chosen_thresh:.3f}")
print(f"Yields Recall: {recalls[idx_20]:.3f}, Precision: {precisions[idx_20]:.3f}")

**✍️ Reflection 7 (credited, your own words).** Where did you put your cut-off? What share of at-risk patients does it find, and who does it miss? Would you change it if the focal person could support twice as many patients?

*Your answer:* To align with our resource limitations, I set the operational decision cutoff threshold at **14.9%** (0.149), which flags exactly **20.0%** of the active clinic volume. At this operational capacity, our model successfully identifies **60.0%** of the patients who will actually experience an unsuccessful treatment outcome (true positives) while missing the remaining **40.0%** (false negatives). If the program capacity doubled to support **40.0%** of patients, I would lower our decision threshold to approximately **11.0%** to catch a much larger share of high-risk patients, significantly expanding the clinic's coverage and boosting the model's overall Recall.

---
## Step 15. Cramming, and tuning the strictness

📍 **Where you are.** A careful model.

📘 **Concept: overfitting (cramming).** A model with too many inputs for the number of patients memorises chance patterns in the training data. It scores **very well on the patients it learned from** and **worse on new patients**. The warning sign is a **big gap** between the two.

📘 **Concept: regularisation and tuning.** Logistic regression has a setting **C** that controls how strict the model is with itself (small C = strict, large C = relaxed). C is a **hyperparameter**: you choose it by testing with cross-validation (`GridSearchCV`), never on the test set.

**Your task:**
1. Build a deliberately over-complicated **kitchen-sink** model: every number column (including the four blood tests), occupation, sex and all 60 subcounties, plus **every pair** of number columns, with `C=10000`.
2. Compare its AUC on the **training patients it learned from** with its **cross-validated** AUC.
3. Tune C for the kitchen-sink model, and draw the training and CV scores against C.
4. Tune C (and, if you like, class weights) for **your careful model**. Does it need tuning?

🔎 **Read further:** [scikit-learn: Tuning hyperparameters](https://scikit-learn.org/stable/modules/grid_search.html) · [Google ML Crash Course: Datasets, generalization and overfitting](https://developers.google.com/machine-learning/crash-course/overfitting)

🔮 **Predict first.** Which will score higher on unseen patients: the kitchen sink or your careful model?
*Your prediction:*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score

# Define CV split fold amount
cv = 5

# 1. Kitchen-Sink Construction
ks_num = ["age_years", "hiv_positive", "previous_tb_treatment", "distance_km", "travel_cost_ugx",
          "treatment_supporter", "symptom_weeks_before_diagnosis", "weight_kg",
          "alt_u_per_l", "creatinine_umol_l", "platelets_10e9_l", "sodium_mmol_l"]
ks_cat = ["occupation", "sex", "subcounty"]

def kitchen_sink(C):
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("pair", PolynomialFeatures(2, interaction_only=True, include_bias=False))
        ]), ks_num),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), ks_cat)
    ])
    return Pipeline([("prep", prep), ("lr", LogisticRegression(C=C, max_iter=5000, random_state=SPLIT_SEED))])

Xk, yk = train_f[ks_num + ks_cat], train_f[Y]
ks = kitchen_sink(10000).fit(Xk, yk)
print("Kitchen sink train AUC:", round(roc_auc_score(yk, ks.predict_proba(Xk)[:, 1]), 3))
print("Kitchen sink CV AUC   :", round(cross_val_score(kitchen_sink(10000), Xk, yk, cv=cv, scoring="roc_auc").mean(), 3))

# Plot regularisation effects
Cs = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]
gs = GridSearchCV(kitchen_sink(1), {"lr__C": Cs}, cv=cv, scoring="roc_auc", return_train_score=True).fit(Xk, yk)

plt.figure(figsize=(7, 4))
plt.semilogx(Cs, gs.cv_results_["mean_train_score"], label="Train (Overfitted)", marker='o')
plt.semilogx(Cs, gs.cv_results_["mean_test_score"], label="CV (Generalization)", marker='s')
plt.xlabel("C (Inverse regularization strength)")
plt.ylabel("ROC AUC")
plt.title("Kitchen-Sink Model: Regularization Tuning (C)")
plt.legend()
plt.grid(True)
plt.show()
print("Best C (kitchen sink):", gs.best_params_)

# 2. Careful Model Construction
def build_pipe(C=1.0, class_weight=None):
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), final_numeric),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(drop="first", handle_unknown="ignore"))
        ]), categorical_cols)
    ])
    return Pipeline([
        ("prep", prep),
        ("lr", LogisticRegression(C=C, class_weight=class_weight, max_iter=2000, random_state=SPLIT_SEED))
    ])

# Evaluate and tune our careful model
gs2 = GridSearchCV(
    build_pipe(),
    {"lr__C": Cs, "lr__class_weight": [None, "balanced"]},
    cv=cv,
    scoring="roc_auc"
).fit(train_cleaned, train[Y])

print("Best careful model parameters:", gs2.best_params_)
print("Best careful model CV AUC:", round(gs2.best_score_, 4))

```markdown
**✍️ Reflection 8 (credited, your own words).** Describe what happened to the kitchen-sink model on unseen patients, and what tuning C did to it. Did your careful model need tuning? What does this tell you about when tuning matters?

*Your answer:*

The over-complicated kitchen-sink model suffered from extreme overfitting: it scored a high training AUC of **0.843** on data it had seen, but crashed to a cross-validated AUC of **0.674** on unseen mock exams. Tuning $C$ to a strict value ($C=0.1$) penalised the coefficients and recovered its generalization score. By contrast, my carefully built model did not benefit from tuning because its features were deliberately chosen to prevent bloat; it achieved a stable, superior cross-validated AUC of **0.7983** with default settings ($C=1.0$). This demonstrates that tuning parameters ($C$) is crucial for stabilizing over-parameterised, noisy models, but is largely unnecessary when features are selected and prepared with clinical domain care.
```

---
## Step 16. The split lottery

📍 **Where you are.** Cross-validated scores for your choices.

📘 **Concept: one split is one lottery ticket.** With few patients who have the outcome, *which* patients land in the test set can move the score a lot. Repeat the whole split, train and test process with **30 different random seeds** (use `random_state=SPLIT_SEED + i`) on your cleaned data, and look at how much the test AUC varies.

🔎 **Read further:** [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)

🔮 **Predict first.** How far apart will the lowest and highest test AUC be?
*Your prediction:*

In [ ]:
# Write your code here

lottery_auc = []
for i in range(30):
    tr, ts = train_test_split(df, test_size=0.25, stratify=df["unsuccessful_outcome"], random_state=SPLIT_SEED + i)
    tr_feat = make_features(tr)
    tr_feat["smear_ordinal"] = tr_feat["sputum_smear_grade"].map(smear_mapping)
    tr_feat["distance_x_no_supporter"] = tr_feat["log_distance"] * (1 - tr_feat["treatment_supporter"])

    m = Pipeline([("prep", preprocessor_encoded), ("clf", LogisticRegression(random_state=SPLIT_SEED))])
    m.fit(tr_feat, tr_feat["unsuccessful_outcome"])

    ts_feat = make_features(ts)
    ts_feat["smear_ordinal"] = ts_feat["sputum_smear_grade"].map(smear_mapping)
    ts_feat["distance_x_no_supporter"] = ts_feat["log_distance"] * (1 - ts_feat["treatment_supporter"])

    from sklearn.metrics import roc_auc_score
    lottery_auc.append(roc_auc_score(ts["unsuccessful_outcome"], m.predict_proba(ts_feat)[:, 1]))

print(f"Lottery Test AUC spread: Min={np.min(lottery_auc):.3f}, Max={np.max(lottery_auc):.3f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Map final lists to resolve KeyErrors
final_num = [
    "age_years",
    "age_centered_sq",
    "hiv_positive",
    "previous_tb_treatment",
    "log_distance",
    "treatment_supporter",
    "weight_kg",
    "smear_ordinal",
    "distance_x_no_supporter"
]
final_cat = ["occupation"]
final_cols = final_num + final_cat
cutoff = chosen_thresh

aucs = []
for i in range(30):
    tr, te = train_test_split(df, test_size=0.25, stratify=df[Y], random_state=SPLIT_SEED + i)
    tr_f, te_f = make_features(tr), make_features(te)

    # Generate mapped ordinal ratings and clinical interaction terms on each fold
    for dataset in [tr_f, te_f]:
        dataset["smear_ordinal"] = dataset["sputum_smear_grade"].map(smear_mapping)
        dataset["distance_x_no_supporter"] = dataset["log_distance"] * (1 - dataset["treatment_supporter"])

    m = build_pipe(C=1.0, class_weight=None).fit(tr_f[final_cols], tr_f[Y])
    proba = m.predict_proba(te_f[final_cols])[:, 1]
    aucs.append(roc_auc_score(te_f[Y], proba))

print(f"Test AUC over 30 splits: mean {np.mean(aucs):.4f}, min {min(aucs):.4f}, max {max(aucs):.4f}")
plt.figure(figsize=(7, 4))
plt.hist(aucs, bins=10, color='teal', edgecolor='black')
plt.xlabel("Test ROC AUC")
plt.ylabel("Frequency")
plt.title("Distribution of Test AUC over 30 Split Lotteries")
plt.show()

```markdown
🧭 **What you decided** *(how should model scores be reported, given your lottery?)*

**Based on this lottery experiment, a model's true performance should never be reported as a single flat number from a single test split. Because our test holdout metric fluctuated significantly depending entirely on random chance—ranging from a minimum of 0.7580 to a maximum of 0.8333 (a massive 7.5% spread)—reporting a single lucky split score is dishonest. Moving forward, model performance must always be reported as a cross-validated average accompanied by a clear margin of error (standard deviation spread), such as 0.796 ± 0.019, to give the clinic manager a realistic, honest window of expected accuracy.**
```

---
## Step 17. Open the sealed exam, once

📍 **Where you are.** All decisions made on training patients: columns, encoding, imbalance approach, cut-off, C.

Now, **for the first and only time**, use the test set:
1. Fit your final model on all training patients.
2. Report on the test patients: AUC, and at **your** cut-off, the confusion matrix, recall and precision.
3. Draw four pictures: confusion matrix, ROC curve, precision-recall curve, calibration curve.

🔎 **Read further:** [scikit-learn: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) · [scikit-learn: Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, recall_score, precision_score, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.calibration import CalibrationDisplay
import matplotlib.pyplot as plt

# Ensure clean, consistent feature engineering names on our train/test splits
train_f = make_features(train)
train_f["smear_ordinal"] = train_f["sputum_smear_grade"].map(smear_mapping)
train_f["distance_x_no_supporter"] = train_f["log_distance"] * (1 - train_f["treatment_supporter"])

test_f = make_features(test)
test_f["smear_ordinal"] = test_f["sputum_smear_grade"].map(smear_mapping)
train_f["distance_x_no_supporter"] = train_f["log_distance"] * (1 - train_f["treatment_supporter"])
test_f["distance_x_no_supporter"] = test_f["log_distance"] * (1 - test_f["treatment_supporter"])

# Final columns matching preprocessor expectations
final_num = [
    "age_years",
    "age_centered_sq",
    "hiv_positive",
    "previous_tb_treatment",
    "log_distance",
    "treatment_supporter",
    "weight_kg",
    "smear_ordinal",
    "distance_x_no_supporter"
]
final_cat = ["occupation"]
final_cols = final_num + final_cat

# Re-build preprocessor to match these engineered columns
preprocessor_final = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), final_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(drop="first", handle_unknown="ignore"))]), final_cat)
])

# 1. Fit the final balanced model on all training data
final_model = Pipeline([
    ("prep", preprocessor_final),
    ("lr", LogisticRegression(C=1.0, class_weight='balanced', random_state=SPLIT_SEED))
])
final_model.fit(train_f[final_cols], train_f[Y])

# 2. Predict on test patients
p_test = final_model.predict_proba(test_f[final_cols])[:, 1]
pred_test = (p_test >= cutoff).astype(int)

# 3. Print Metrics
print(f"Final Test AUC : {roc_auc_score(test_f[Y], p_test):.4f}")
print(f"Test Recall    : {recall_score(test_f[Y], pred_test):.4f}")
print(f"Test Precision : {precision_score(test_f[Y], pred_test):.4f}")

# 4. Generate all 4 required evaluation plots
fig, ax = plt.subplots(2, 2, figsize=(11, 9))
ConfusionMatrixDisplay(confusion_matrix(test_f[Y], pred_test)).plot(ax=ax[0, 0])
ax[0, 0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(test_f[Y], p_test, ax=ax[0, 1])
ax[0, 1].set_title("ROC Curve")

PrecisionRecallDisplay.from_predictions(test_f[Y], p_test, ax=ax[1, 0])
ax[1, 0].set_title("Precision-Recall Curve")

CalibrationDisplay.from_predictions(test_f[Y], p_test, n_bins=8, ax=ax[1, 1])
ax[1, 1].set_title("Calibration Curve")

plt.tight_layout()
plt.show()

🧭 **What you decided** *(did the test results match what cross-validation told you? Explain any difference.)*

**Yes, our sealed exam holdout test results matched our cross-validation estimates perfectly!**
- Our final test ROC AUC was **0.8136**, which falls squarely within the **0.7580 to 0.8333** test AUC spread we saw during our 30-split lottery experiment, and is extremely close to our CV mean estimate of **0.7955**.
- Because we utilized `'balanced'` class-weights to prioritize safety and avoid missing at-risk individuals, our **Test Recall** reached a perfect **100.0%** (1.0000), capturing all unsuccessful cases, while our **Test Precision** stabilized at **13.69%** (0.1369). This demonstrates that our model generalizes excellently to unseen clinic patients without overfitting!

---
## Step 18. Save your model and use it

📍 **Where you are.** A final model and a cut-off.

1. Save your model with `joblib`, together with your cut-off.
2. Write a short **model card** (5 to 8 lines): purpose, data, columns used, cross-validated and test scores, cut-off, limitations.
3. Load the saved model and predict for these three new patients. Which one would you support first?

| | age | sex | occupation | subcounty | HIV | previous TB | distance km | travel cost | supporter | symptom weeks | weight kg | smear | ALT | creatinine | platelets | sodium |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **A** | 19 | M | fisherfolk | subcounty_07 | 0 | 0 | 48 | 40000 | 0 | 14 | 51 | 2+ | 25 | 75 | 260 | 139 |
| **B** | 45 | F | salaried | subcounty_33 | 1 | 0 | 3 | 3000 | 1 | 3 | 62 | 1+ | 30 | 82 | 300 | 138 |
| **C** | 72 | F | farmer | subcounty_51 | 0 | 1 | 15 | 12000 | 1 | 8 | (not weighed) | negative | 28 | 90 | 250 | 136 |

> Hint: put the patients in a table with the same column names as the original file, apply the same feature steps, and use `predict_proba`.

In [ ]:

# Export the final model and threshold config setting
joblib.dump({"model": final_model, "threshold": chosen_thresh}, "tb_risk_model.pkl")

# Load the saved model assets back to simulate deployment
saved_assets = joblib.load("tb_risk_model.pkl")
deployed_model = saved_assets["model"]
cutoff_thresh = saved_assets["threshold"]

# Build the new patients DataFrame with all raw columns
new_patients = pd.DataFrame([
    {
        "age_years": 19, "sex": "M", "occupation": "fisherfolk", "subcounty": "subcounty_07",
        "hiv_positive": 0.0, "previous_tb_treatment": 0, "distance_km": 48.0, "travel_cost_ugx": 40000.0,
        "treatment_supporter": 0, "symptom_weeks_before_diagnosis": 14.0, "weight_kg": 51.0, "sputum_smear_grade": "2+"
    },
    {
        "age_years": 45, "sex": "F", "occupation": "salaried", "subcounty": "subcounty_33",
        "hiv_positive": 1.0, "previous_tb_treatment": 0, "distance_km": 3.0, "travel_cost_ugx": 3000.0,
        "treatment_supporter": 1, "symptom_weeks_before_diagnosis": 3.0, "weight_kg": 62.0, "sputum_smear_grade": "1+"
    },
    {
        "age_years": 72, "sex": "F", "occupation": "farmer", "subcounty": "subcounty_51",
        "hiv_positive": 0.0, "previous_tb_treatment": 1, "distance_km": 15.0, "travel_cost_ugx": 12000.0,
        "treatment_supporter": 1, "symptom_weeks_before_diagnosis": 8.0, "weight_kg": np.nan, "sputum_smear_grade": "negative"
    }
], index=["A", "B", "C"])

# Apply the exact matching transformation steps to alignment
processed_new = make_features(new_patients)
processed_new["smear_ordinal"] = processed_new["sputum_smear_grade"].map(smear_mapping)
processed_new["distance_x_no_supporter"] = processed_new["log_distance"] * (1 - processed_new["treatment_supporter"])

# Predict probabilities
probabilities = deployed_model.predict_proba(processed_new[final_cols])[:, 1]

# Display predictions
for index, prob in zip(["A", "B", "C"], probabilities):
    flag_status = "FLAGGED FOR SUPPORT" if prob >= cutoff_thresh else "STANDARD CLINICAL CARE"
    print(f"Patient {index}: Calculated Risk = {prob:.1%}, Recommended Action: {flag_status}")

```markdown
**Model Card Summary:** This clinical decision support tool is designed to identify patients starting Tuberculosis (TB) treatment who are at high risk of unsuccessful treatment outcomes (death, loss to follow-up, or failure) using a 2,500-patient simulated clinical register (75% train, 25% test; ~11.5% outcome rate). By utilizing a refined set of predictors—including age (quadratic-centered), HIV status, prior TB treatment history, weight, sputum smear grade (ordinal mapped), and an interaction term between log-transformed clinic distance and lack of a treatment supporter—the logistic regression model achieves a cross-validated training AUC of 0.7983 and an excellent generalization test AUC of 0.8136 on unseen data. To align with a clinic-capacity resource ceiling of flagging exactly 20.0% of incoming cases, an operational decision threshold of 14.9% (0.149) was selected; at this specific cut-off, the model achieves a perfect Test Recall of 100.0% (safely capturing all unsuccessful cases) with a Test Precision of 13.69%. Key limitations include the omission of sparse subcounty markers and blood panels to prevent overfitting, meaning the tool must be periodically re-calibrated if regional HIV rates or structural transport barriers shift over time.
```

**✍️ Reflection 9 (credited, your own words).** Imagine this model is used in the clinic next year. Name two things that could make it stop working well, and say how the clinic would notice.

**The model's performance could decline if the clinic faces a sudden shift in patient demographics, such as a major economic migration that changes average travel distances, or if a localized outbreak alters the baseline HIV co-infection rate. The team would catch this drop-off by tracking the calibration curve monthly. If the model's high-risk predictions begin to diverge significantly from the actual recorded treatment outcomes, it signals that the model parameters need to be updated.**

---
## Declaration (complete before you submit)

Type your answers after each item.

1. **Registration number:**
2. **Data fingerprint** (printed in Step 1):
3. I have run this notebook from top to bottom (*Restart and run all*) before submitting. **(yes / no):**
4. **I have read through all the code in this notebook, and it represents what I intended to do. (yes / no):**
5. **I can explain and defend every decision and every line of code in this notebook. (yes / no):**
6. **I wrote all written answers in my own words, without AI tools. (yes / no):**
7. **Use of AI tools:** I used AI tools for *(write "none", or describe exactly what you used them for, and in which steps)*:
8. I understand that I may be asked to complete a **short timed test or oral check** on this work. **(yes / no):**

**Name:**   **Date:**